# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
# The IBUG cells later change directory (%cd ibug, %cd .., %cd ..) and then use
# "./content/drive/..." and "/content/drive/..." (Colab layout); on Kaggle both are linked
# to /kaggle/working so every Excel file lands in the same output folder.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d05_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # concrete train
    shutil.copy(glob.glob("/kaggle/input/**/d05_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # concrete test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: concrete data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: concrete data copied to ./drive/MyDrive/Regression/concrete_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795359809.050876      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795359827.738539      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (concrete_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']
TARGET_LABEL = 'Compressive strength (str)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [7]:
train_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title(f'Predicted {TARGET_LABEL} vs. Actual {TARGET_LABEL}')
    plt.xlabel(f'Actual {TARGET_LABEL}')
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel(f'Predicted Mean {TARGET_LABEL}')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label=f'Actual {TARGET_LABEL}',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label=f'Predicted Mean {TARGET_LABEL}',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/concrete-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 11.477140958909033,
  'test_rmse': np.float64(3.3877929332987624),
  'test_corr_coef': np.float64(0.9430872254495873),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease

In [20]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 11.477140958909033,
  'test_rmse': np.float64(3.3877929332987624),
  'test_corr_coef': np.float64(0.9430872254495873),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 19.61800683683138,
  'cv_rmse': 4.429221922282895,
  'best_params': {'n_estimators': 500,
   'criterion': 'friedman_mse',
   'max_depth': 30,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'log2',
   'max_leaf_nodes': None,
   'min_impurity_decrease

# **Probabilistic distributions using NGBoost**

In [21]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize NGBoost model with best_params
ngb_model = NGBRegressor(Dist=Normal, **best_params)

# Fit the model
ngb_model.fit(X_train, y_train)

# Predict the distribution
pred_dist = ngb_model.pred_dist(X_test)

# Extract mean and standard deviation
predictions_NGB_df = pd.DataFrame({
    'Mean': pred_dist.loc,
    'StdDev': pred_dist.scale,
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, pred_dist.loc)
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_NGB_df.head())

Mean Squared Error: 9.93
        Mean    StdDev  Actual
0  32.798216  0.406402    34.0
1  35.588986  0.347432    36.0
2  33.743115  0.851331    36.0
3  36.201605  0.293772    35.0
4  44.375882  0.108995    44.6


In [22]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


In [23]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,1.869266,0.410451,3.955879,5.415125,9


In [24]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,PIT_Value
0,0.998447
1,0.881597
2,0.995987
3,0.000022
4,0.980120
...,...
90,0.000000
91,0.906609
92,0.999227
93,0.086271


In [25]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.347368,0.247368
2,0.2,0.400000,0.200000
3,0.3,0.410526,0.110526
4,0.4,0.431579,0.031579
5,0.5,0.452632,0.047368
6,0.6,0.473684,0.126316
7,0.7,0.494737,0.205263
8,0.8,0.568421,0.231579
9,0.9,0.610526,0.289474


# **Probabilistic distributions using PGBM**

In [26]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
# Note: Adjust the incompatible keys based on PGBM's requirements
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params = best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Extract mean and standard deviation
# Note: PGBM might return predictions in a different format, adjust accordingly
predictions_PGBM_df = pd.DataFrame({
    'Mean': pred_dist.mean(axis=0),
    'StdDev': pred_dist.std(axis=0),
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, predictions_PGBM_df['Mean'])
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 9.3540
Estimator 1/300, Train metric: 8.7014
Estimator 2/300, Train metric: 8.2176
Estimator 3/300, Train metric: 7.7928
Estimator 4/300, Train metric: 7.4588
Estimator 5/300, Train metric: 7.0766
Estimator 6/300, Train metric: 6.7824
Estimator 7/300, Train metric: 6.3936
Estimator 8/300, Train metric: 6.0929
Estimator 9/300, Train metric: 5.8563
Estimator 10/300, Train metric: 5.5902


Estimator 11/300, Train metric: 5.3150
Estimator 12/300, Train metric: 5.1076
Estimator 13/300, Train metric: 4.9502
Estimator 14/300, Train metric: 4.7788
Estimator 15/300, Train metric: 4.6107
Estimator 16/300, Train metric: 4.4217
Estimator 17/300, Train metric: 4.2602
Estimator 18/300, Train metric: 4.0980
Estimator 19/300, Train metric: 3.9577
Estimator 20/300, Train metric: 3.8037
Estimator 21/300, Train metric: 3.6834
Estimator 22/300, Train metric: 3.5716
Estimator 23/300, Train metric: 3.4367
Estimator 24/300, Train metric: 3.3685
Estimator 25/300, Train metric: 3.2836
Estimator 26/300, Train metric: 3.1784


Estimator 27/300, Train metric: 3.1301
Estimator 28/300, Train metric: 3.0346
Estimator 29/300, Train metric: 2.9881
Estimator 30/300, Train metric: 2.9410
Estimator 31/300, Train metric: 2.8818
Estimator 32/300, Train metric: 2.8382
Estimator 33/300, Train metric: 2.7729
Estimator 34/300, Train metric: 2.7350
Estimator 35/300, Train metric: 2.6757
Estimator 36/300, Train metric: 2.6177
Estimator 37/300, Train metric: 2.5424
Estimator 38/300, Train metric: 2.5075
Estimator 39/300, Train metric: 2.4696
Estimator 40/300, Train metric: 2.4056
Estimator 41/300, Train metric: 2.3744
Estimator 42/300, Train metric: 2.3309


Estimator 43/300, Train metric: 2.2890
Estimator 44/300, Train metric: 2.2685
Estimator 45/300, Train metric: 2.2192
Estimator 46/300, Train metric: 2.2087
Estimator 47/300, Train metric: 2.1725
Estimator 48/300, Train metric: 2.1472
Estimator 49/300, Train metric: 2.1308
Estimator 50/300, Train metric: 2.0869
Estimator 51/300, Train metric: 2.0626
Estimator 52/300, Train metric: 2.0371
Estimator 53/300, Train metric: 2.0172
Estimator 54/300, Train metric: 1.9750
Estimator 55/300, Train metric: 1.9376
Estimator 56/300, Train metric: 1.9247
Estimator 57/300, Train metric: 1.8948


Estimator 58/300, Train metric: 1.8667
Estimator 59/300, Train metric: 1.8407
Estimator 60/300, Train metric: 1.8067
Estimator 61/300, Train metric: 1.7990
Estimator 62/300, Train metric: 1.7828
Estimator 63/300, Train metric: 1.7553
Estimator 64/300, Train metric: 1.7234
Estimator 65/300, Train metric: 1.7172
Estimator 66/300, Train metric: 1.6990
Estimator 67/300, Train metric: 1.6698
Estimator 68/300, Train metric: 1.6585
Estimator 69/300, Train metric: 1.6427
Estimator 70/300, Train metric: 1.6199
Estimator 71/300, Train metric: 1.6148
Estimator 72/300, Train metric: 1.5931
Estimator 73/300, Train metric: 1.5767
Estimator 74/300, Train metric: 1.5664


... (11 outputs left out)

Estimator 254/300, Train metric: 0.6958
Estimator 255/300, Train metric: 0.6950
Estimator 256/300, Train metric: 0.6937
Estimator 257/300, Train metric: 0.6924
Estimator 258/300, Train metric: 0.6910
Estimator 259/300, Train metric: 0.6915
Estimator 260/300, Train metric: 0.6902
Estimator 261/300, Train metric: 0.6865
Estimator 262/300, Train metric: 0.6858
Estimator 263/300, Train metric: 0.6851
Estimator 264/300, Train metric: 0.6847
Estimator 265/300, Train metric: 0.6830
Estimator 266/300, Train metric: 0.6815
Estimator 267/300, Train metric: 0.6809
Estimator 268/300, Train metric: 0.6813


Estimator 269/300, Train metric: 0.6801
Estimator 270/300, Train metric: 0.6788
Estimator 271/300, Train metric: 0.6774
Estimator 272/300, Train metric: 0.6748
Estimator 273/300, Train metric: 0.6747
Estimator 274/300, Train metric: 0.6742
Estimator 275/300, Train metric: 0.6732
Estimator 276/300, Train metric: 0.6701
Estimator 277/300, Train metric: 0.6680
Estimator 278/300, Train metric: 0.6676
Estimator 279/300, Train metric: 0.6642
Estimator 280/300, Train metric: 0.6630
Estimator 281/300, Train metric: 0.6630
Estimator 282/300, Train metric: 0.6619
Estimator 283/300, Train metric: 0.6608


Estimator 284/300, Train metric: 0.6602
Estimator 285/300, Train metric: 0.6581
Estimator 286/300, Train metric: 0.6562
Estimator 287/300, Train metric: 0.6562
Estimator 288/300, Train metric: 0.6535
Estimator 289/300, Train metric: 0.6510
Estimator 290/300, Train metric: 0.6493
Estimator 291/300, Train metric: 0.6458
Estimator 292/300, Train metric: 0.6446
Estimator 293/300, Train metric: 0.6449
Estimator 294/300, Train metric: 0.6438
Estimator 295/300, Train metric: 0.6426
Estimator 296/300, Train metric: 0.6412
Estimator 297/300, Train metric: 0.6405
Estimator 298/300, Train metric: 0.6394
Estimator 299/300, Train metric: 0.6384


Mean Squared Error: 9.18
        Mean    StdDev  Actual
0  33.058258  0.367396    34.0
1  36.749100  0.284790    36.0
2  34.798588  0.351820    36.0
3  36.891930  0.374452    35.0
4  42.851196  0.574225    44.6


In [27]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


In [28]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,1.924366,0.585338,4.075601,5.415125,9


In [29]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,PIT_Value
0,9.948157e-01
1,4.264789e-03
2,9.996809e-01
3,2.179934e-07
4,9.988385e-01
...,...
90,1.836082e-48
91,3.193162e-01
92,9.999875e-01
93,3.243486e-01


In [30]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.368421,0.268421
2,0.2,0.368421,0.168421
3,0.3,0.389474,0.089474
4,0.4,0.442105,0.042105
5,0.5,0.452632,0.047368
6,0.6,0.494737,0.105263
7,0.7,0.526316,0.173684
8,0.8,0.536842,0.263158
9,0.9,0.547368,0.352632


# **Probabilistic Distribution using GPBoost (Treeffuser)**


In [31]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_score_engine = TreeBasedScoreModel(
    model_name='gpboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

# The fix is in the line below: '_score_model' has been corrected to 'score_model'.
model = Treeffuser(score_model=gpboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GPBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_TF_df.head())


[1]	valid_0's l2: 0.824776
[2]	valid_0's l2: 0.722232
[3]	valid_0's l2: 0.653507
[4]	valid_0's l2: 0.594014
[5]	valid_0's l2: 0.557171
[6]	valid_0's l2: 0.555582
[7]	valid_0's l2: 0.529223
[8]	valid_0's l2: 0.53197
[9]	valid_0's l2: 0.531386
[10]	valid_0's l2: 0.531661
[11]	valid_0's l2: 0.515047
[12]	valid_0's l2: 0.515015
[13]	valid_0's l2: 0.515509
[14]	valid_0's l2: 0.509578
[15]	valid_0's l2: 0.511934
[16]	valid_0's l2: 0.520814
[17]	valid_0's l2: 0.520296
[18]	valid_0's l2: 0.522073
[19]	valid_0's l2: 0.521716
[20]	valid_0's l2: 0.524866
[21]	valid_0's l2: 0.520541
[22]	valid_0's l2: 0.518761
[23]	valid_0's l2: 0.51352
[24]	valid_0's l2: 0.514418
[25]	valid_0's l2: 0.512844
[26]	valid_0's l2: 0.520612
[27]	valid_0's l2: 0.520689
[28]	valid_0's l2: 0.521428
[29]	valid_0's l2: 0.527178
[30]	valid_0's l2: 0.530114
[31]	valid_0's l2: 0.538443
[32]	valid_0's l2: 0.538894
[33]	valid_0's l2: 0.540137
[34]	valid_0's l2: 0.541028
[35]	valid_0's l2: 0.538606
[36]	valid_0's l2: 0.539899
[37

        Mean     StdDev  Actual
0  35.593406   5.292096    34.0
1  34.393352  11.665736    36.0
2  35.626190   5.572130    36.0
3  29.874201   4.372664    35.0
4  48.459345   7.515841    44.6


In [32]:
generate_and_save_plots(predictions_GPBoost_TF_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


In [33]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,2.464634,0.175676,3.125855,5.415125,9


In [34]:
generate_and_save_pit_results(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,PIT_Value
0,0.381672
1,0.554771
2,0.526743
3,0.879449
4,0.303803
...,...
90,0.176383
91,0.907793
92,0.575000
93,0.419425


In [35]:
generate_and_save_calibration_curve(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.010526,0.089474
2,0.2,0.063158,0.136842
3,0.3,0.168421,0.131579
4,0.4,0.336842,0.063158
5,0.5,0.484211,0.015789
6,0.6,0.621053,0.021053
7,0.7,0.800000,0.100000
8,0.8,0.905263,0.105263
9,0.9,0.968421,0.068421


# **Probabilistic Distribution using Gradient Boosting (Treeffuser)**

In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_score_engine = TreeBasedScoreModel(
    model_name='gradient_boosting',
    # FIX T1: the diffusion score model is fitted by least squares (Treeffuser's denoising
    # objective); the tuned GBR loss may be absolute_error/huber, so force squared_error here.
    model_params={**best_params, 'loss': 'squared_error'},
    seed=42,
)

model = Treeffuser(score_model=gbr_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GBR_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_TF_df.head())


        Mean    StdDev  Actual
0  35.015256  5.547160    34.0
1  33.573455  4.857055    36.0
2  32.979615  7.276720    36.0
3  30.122151  4.332500    35.0
4  54.721719  8.057615    44.6


In [37]:
generate_and_save_plots(predictions_GBR_TF_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


In [38]:
generate_and_save_crps_decomposition(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="Gradient Boosting",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,2.475337,0.08507,3.024449,5.415125,9


In [39]:
generate_and_save_pit_results(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,PIT_Value
0,0.427390
1,0.691319
2,0.660957
3,0.869891
4,0.104528
...,...
90,0.025430
91,0.775586
92,0.638101
93,0.328197


In [40]:
generate_and_save_calibration_curve(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.031579,0.068421
2,0.2,0.073684,0.126316
3,0.3,0.252632,0.047368
4,0.4,0.357895,0.042105
5,0.5,0.505263,0.005263
6,0.6,0.673684,0.073684
7,0.7,0.778947,0.078947
8,0.8,0.863158,0.063158
9,0.9,0.936842,0.036842


# **Probabilistic Distribution using Catboost (Treeffuser)**

In [41]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_score_engine = TreeBasedScoreModel(
    model_name='catboost',
    model_params=best_params,
    seed=42,
)

model = Treeffuser(score_model=catboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_CatBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_TF_df.head())

        Mean      StdDev  Actual
0  54.416404   94.297971    34.0
1  33.360285  105.006553    36.0
2  36.269276  112.044718    36.0
3  52.108944  104.578883    35.0
4  59.515661  103.397301    44.6


In [42]:
generate_and_save_plots(predictions_CatBoost_TF_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


In [43]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,23.332773,17.503624,-0.418607,5.415125,9


In [44]:
generate_and_save_pit_results(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,PIT_Value
0,0.414295
1,0.510028
2,0.499041
3,0.435024
4,0.442649
...,...
90,0.414614
91,0.515586
92,0.509291
93,0.505466


In [45]:
generate_and_save_calibration_curve(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.021053,0.378947
5,0.5,0.768421,0.268421
6,0.6,0.989474,0.389474
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using XGBoost (Treeffuser)**

In [46]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_score_engine = TreeBasedScoreModel(
    model_name='xgboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=xgb_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_XGB_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_TF_df.head())

[0]	validation_0-rmse:0.83978


[1]	validation_0-rmse:0.79267


[2]	validation_0-rmse:0.75256


[3]	validation_0-rmse:0.73262


[4]	validation_0-rmse:0.73949


[5]	validation_0-rmse:0.74363


... (491 outputs left out)

[497]	validation_0-rmse:0.76772


[498]	validation_0-rmse:0.76772


[499]	validation_0-rmse:0.76784

        Mean     StdDev  Actual
0  34.230027  21.909111    34.0
1  35.202676  17.518755    36.0
2  35.399225  16.758285    36.0
3  32.239883  23.567189    35.0
4  64.348748  33.342321    44.6


In [47]:
generate_and_save_plots(predictions_XGB_TF_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


In [48]:
generate_and_save_crps_decomposition(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,6.153859,2.183191,1.443406,5.415125,9


In [49]:
generate_and_save_pit_results(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,PIT_Value
0,0.495812
1,0.518151
2,0.514299
3,0.546616
4,0.276824
...,...
90,0.366621
91,0.594539
92,0.472663
93,0.448388


In [50]:
generate_and_save_calibration_curve(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.010526,0.189474
3,0.3,0.052632,0.247368
4,0.4,0.252632,0.147368
5,0.5,0.589474,0.089474
6,0.6,0.831579,0.231579
7,0.7,0.968421,0.268421
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using LGBM (Treeffuser)**

In [51]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_score_engine = TreeBasedScoreModel(
    model_name='lightgbm',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=lgbm_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_LGBM_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_TF_df.head())


        Mean     StdDev  Actual
0  36.233451   8.802838    34.0
1  35.593825   9.720673    36.0
2  35.470228   7.642942    36.0
3  29.938343   5.835512    35.0
4  56.739169  18.177032    44.6


In [52]:
generate_and_save_plots(predictions_LGBM_TF_df, y_test, "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


In [53]:
generate_and_save_crps_decomposition(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,3.271325,0.591344,2.734689,5.415125,9


In [54]:
generate_and_save_pit_results(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,PIT_Value
0,0.399856
1,0.516665
2,0.527631
3,0.807135
4,0.252121
...,...
90,0.323427
91,0.831918
92,0.483330
93,0.523011


In [55]:
generate_and_save_calibration_curve(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.010526,0.089474
2,0.2,0.031579,0.168421
3,0.3,0.126316,0.173684
4,0.4,0.347368,0.052632
5,0.5,0.547368,0.047368
6,0.6,0.736842,0.136842
7,0.7,0.894737,0.194737
8,0.8,0.936842,0.136842
9,0.9,0.978947,0.078947


# **IBUG Installation**

In [56]:
# Clone the repository
!git clone https://github.com/jjbrophy47/ibug.git
%cd ibug

# Build extensions if needed
!python setup.py build_ext --inplace

# Install the package
!pip install .

# Verify installation
try:
    from ibug import IBUGWrapper
    print("IBUGWrapper imported successfully!")
except ImportError as e:
    print("Error importing IBUGWrapper:", e)

Cloning into 'ibug'...


remote: Enumerating objects: 1763, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.


remote: Total 1763 (delta 18), reused 15 (delta 4), pack-reused 1726 (from 1)


Receiving objects: 100% (1763/1763), 1.31 MiB | 13.66 MiB/s, done.


Resolving deltas: 100% (967/967), done.


/kaggle/working/ibug


... (18 outputs left out)

  Preparing metadata (pyproject.toml) ... done


  Created wheel for ibug: filename=ibug-0.0.12-cp313-cp313-linux_x86_64.whl size=1581580 sha256=bb6552ad33cb2744e7fa3e2377dbf99996b9f212df90f53ddd7009bbecde4f1a
  Stored in directory: /tmp/pip-ephem-wheel-cache-ziewe_xa/wheels/48/c2/c1/d39bc91440bc7e7da40b146580d8aeb84c5361fa857fe8037e
Successfully built ibug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ibug]


IBUGWrapper imported successfully!


In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)  # FIX P2: seeded


# **Probabilistic Distributions using LGBM (IBUG)**

In [58]:
# y_train = y_train.to_numpy()
# y_val = y_val.to_numpy()

In [59]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# --- Ensure IBUG compatibility ---
best_params['reg_alpha'] = 0
best_params['class_weight'] = None
best_params['boosting_type'] = 'gbdt'

# Initialize and train the LightGBM regressor using the best_params
model = LGBMRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location2, scale2, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale**0.5
# Create a DataFrame for clarity
predictions_LGBM_df = pd.DataFrame({
    'Mean': location,
    'StdDev': dev,
    'Actual': y_test,
})

print(predictions_LGBM_df.head())

        Mean    StdDev  Actual
0  32.333833  3.863792    34.0
1  32.476490  2.515287    36.0
2  33.246989  2.328567    36.0
3  32.633074  2.515287    35.0
4  47.609013  4.788822    44.6


In [60]:
%cd ..

/kaggle/working


In [61]:
%cd ..

/kaggle


In [62]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx")

Excel file saved at ./content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


In [63]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx
   Sheets: CRPS_Summary_LGBM, CRPS_Bins_LGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LGBM,1.803579,0.011262,3.622544,5.415125,9


In [64]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,PIT_Value
0,0.666848
1,0.919369
2,0.881452
3,0.826652
4,0.264890
...,...
90,0.376110
91,0.750354
92,0.673858
93,0.385534


In [65]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.115789,0.015789
2,0.2,0.200000,0.000000
3,0.3,0.284211,0.015789
4,0.4,0.389474,0.010526
5,0.5,0.463158,0.036842
6,0.6,0.568421,0.031579
7,0.7,0.726316,0.026316
8,0.8,0.789474,0.010526
9,0.9,0.894737,0.005263


# **Probabilistic Distributions using XGBoost (IBUG)**

In [66]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

best_params['reg_alpha'] = 0
best_params['scale_pos_weight'] = 1
best_params['base_score'] = 0.5

# Initialize and train the XGBoost regressor using the best_params
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
# This returns the predicted mean (location) and uncertainty (scale, variance proxy)
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location, scale, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

# Calculate the standard deviation from the variance (scale)
std_dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale ** 0.5

# Create a DataFrame for clarity
predictions_XGB_df = pd.DataFrame({
    'Mean': location,
    'StdDev': std_dev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

        Mean    StdDev  Actual
0  33.030258  5.191612    34.0
1  32.932491  5.389281    36.0
2  33.823231  3.604990    36.0
3  32.784863  5.389281    35.0
4  45.464203  9.164926    44.6


In [67]:
generate_and_save_plots(predictions_XGB_df, y_test, "./content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx")  # FIX P3: was predictions_LGBM_df

Excel file saved at ./content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


In [68]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx
   Sheets: CRPS_Summary_XGB, CRPS_Bins_XGB


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGB,1.836164,0.162185,3.740918,5.415125,9


In [69]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,PIT_Value
0,0.574087
1,0.715385
2,0.727019
3,0.659473
4,0.462438
...,...
90,0.281868
91,0.568707
92,0.671718
93,0.364304


In [70]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.021053,0.078947
2,0.2,0.073684,0.126316
3,0.3,0.189474,0.110526
4,0.4,0.315789,0.084211
5,0.5,0.505263,0.005263
6,0.6,0.684211,0.084211
7,0.7,0.800000,0.100000
8,0.8,0.905263,0.105263
9,0.9,0.947368,0.047368


# **Matrix Evaulation**

In [71]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/concrete_uncertainty_analysis/probabilistic_distribution/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_TF_df, y_test, "LightGBM_TF")
    plot_metrics_and_save_to_excel(writer, predictions_XGB_TF_df, y_test, "XGBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_TF_df, y_test, "CatBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_TF_df, y_test, "GPBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_TF_df, y_test, "GradientBoosting_TF")